# 03 · Portfolio agent
Documentation: `docs/agents/portfolio_agent.md`. Diagnostic files are written at every agent run.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_rows', 80); pd.set_option('display.width', 200)
D = ROOT / 'data'
def ok(p):
    if not Path(p).exists(): print('File not available yet:', p); return False
    return True
P = D/'portfolio'
from core.config import load_config
from agents.portfolio_agent.agent import apply_profile
from agents.portfolio_agent.backtest import regime_profile
cfg = load_config(); pcfg = regime_profile(apply_profile(dict(cfg['portfolio'])), 'offense')
uni = pd.read_parquet(D/'processed/universe.parquet')

## 1. Effective limits (base → growth → offense)

In [ ]:
for k in ['class_caps','equity_total_cap','region_caps','ai_cap','hedge','risk','ml','ml_returns','mc_selection','mc_scenarios','choose_by']:
    print(f'{k:18}', pcfg[k] if k != 'hedge' else {kk: v for kk, v in pcfg[k].items() if kk != 'tickers'})

## 2. Backtest metrics: full period and by horizon

In [ ]:
m = pd.read_csv(P/'backtest_metrics.csv', index_col=0); display(m.round(3))
hz = {f.stem.replace('metrics_',''): pd.read_csv(f, index_col=0) for f in sorted(P.glob('metrics_*.csv'))}
pd.DataFrame({h: x['Return'] for h, x in hz.items()}).round(3)

## 3. Equity curves and drawdowns

In [ ]:
r = pd.read_parquet(P/'backtest_returns.parquet'); eq = (1+r.fillna(0)).cumprod()
fig, ax = plt.subplots(2, 1, figsize=(13,8), sharex=True, gridspec_kw={'height_ratios':[2,1]})
eq.plot(ax=ax[0], logy=True); ax[0].set_title('Value (base 1, EUR, log scale)')
(eq/eq.cummax()-1).plot(ax=ax[1], legend=False); ax[1].set_title('Drawdown'); plt.show()

In [ ]:
y = ((1+r).resample('YE').prod()-1); y.index = y.index.year; (y*100).round(1)

## 4. Weights by asset class over time (choose the method)

In [ ]:
METHOD = 'max_sharpe'
if ok(P/'class_weights_history.csv'):
    cw = pd.read_csv(P/'class_weights_history.csv', index_col=[0,1], parse_dates=[1]).loc[METHOD]
    cw.plot.area(figsize=(13,4), title=f'{METHOD}: weights by asset class'); plt.legend(bbox_to_anchor=(1,1)); plt.show()

In [ ]:
wh = pd.read_parquet(P/'weights_history.parquet')
last = wh.loc[METHOD].dropna(axis=1, how='all').tail(4).T
last[last.notna().any(axis=1)].fillna(0).sort_values(last.columns[-1], ascending=False).round(3).head(30)

## 5. Turnover and costs

In [ ]:
w = wh.loc[METHOD].fillna(0)
to = w.diff().abs().sum(axis=1); trades = (w.diff().abs() > 1e-4).sum(axis=1)
pd.DataFrame({'turnover': to.round(2), 'orders': trades, 'cost_%': (to*pcfg['cost_bps']/1e4 + trades*pcfg['commission_per_trade']/pcfg['capital'])*100}).tail(12).round(2)

## 6. Factors: predictive power (IC) and weights used over time

In [ ]:
ic = pd.read_parquet(P/'factor_ic.parquet')
s = ic.groupby(['family','factor']).ic.agg(['mean','std','count']); s['ICIR'] = s['mean']/s['std']
display(s.sort_values(['family','mean'], ascending=[True,False]).round(3))
roll = ic[ic.family=='equity'].pivot(index='date', columns='factor', values='ic').rolling(24).mean()
roll[['mom_12_1','risk_adj_mom','low_vol','gross_prof','fcf_yield','earnings_yield']].plot(figsize=(13,4), title='24-month rolling mean IC (equities)'); plt.axhline(0, color='k', lw=.5); plt.show()

In [ ]:
if ok(P/'factor_weights_history.csv'):
    fw = pd.read_csv(P/'factor_weights_history.csv', parse_dates=['date'])
    h = fw[fw.family=='equity'].pivot_table(index='date', columns='factor', values='weight').fillna(0)
    h.plot.area(figsize=(13,4), title='Factor weights used (equities)'); plt.legend(bbox_to_anchor=(1,1)); plt.show()

## 7. The two LightGBM models: out-of-sample quality in the backtest

In [ ]:
if ok(P/'ml_oos_ic.csv'):
    o = pd.read_csv(P/'ml_oos_ic.csv', parse_dates=['date'])
    display(o.groupby('model').ic.agg(['mean','count', lambda x: (x>0).mean()]).rename(columns={'<lambda_0>':'share_positive'}).round(3))
    o.pivot_table(index='date', columns='model', values='ic').rolling(8, min_periods=4).mean().plot(figsize=(13,4), title='Out-of-sample IC (rolling mean)'); plt.axhline(0, color='k', lw=.5); plt.show()
p = D/'experiments/ml_tuning.csv'
if ok(p): display(pd.read_csv(p).head(10).round(4))

## 8. Context signals over time (AI bubble, high volatility, mode)

In [ ]:
if ok(P/'context_history.csv'):
    ch = pd.read_csv(P/'context_history.csv', index_col=0, parse_dates=True)
    display(ch.astype(str).apply(lambda c: c.value_counts()).fillna(0))
    ch[[c for c in ['bubble','high_vol'] if c in ch]].astype(str).replace({'True':1,'False':0}).astype(float).plot(figsize=(13,2.5), drawstyle='steps-post'); plt.show()

## 9. Today: scores, expected returns, method selection, proposal

In [ ]:
if ok(P/'scores_today.csv'):
    sc = pd.read_csv(P/'scores_today.csv', index_col=0); display(sc.head(30).round(3))
if ok(P/'expected_returns_ml.csv'):
    mu = pd.read_csv(P/'expected_returns_ml.csv', index_col=0).iloc[:,0].sort_values()
    print('LightGBM expected returns (relative, annualised): lowest and highest'); display(pd.concat([mu.head(10), mu.tail(10)]).to_frame().join(uni[['asset_class','name']]).round(3))

In [ ]:
display(pd.read_csv(P/'mc_selection.csv', index_col=0).round(3))
tg = pd.read_csv(P/'target_portfolio.csv', index_col=0); display(tg.round(3))
print(json.dumps(json.loads((P/'context.json').read_text()), indent=1, default=str)[:3000])